### Loading CSV files into Postgresql

In [ ]:
# importing libraries for data loading

In [1]:
!pip install sqlalchemy psycopg2-binary

Defaulting to user installation because normal site-packages is not writeable


In [5]:
import pandas as pd
import os
from sqlalchemy import create_engine,text

In [ ]:
#Connect to PostgreSQL

In [3]:
username = "postgres"
password = "password"
host = "localhost"
port = "5432"
database = "technova_finance"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

with engine.connect() as conn:
    print("Connected Successfully!")

Connected Successfully!


In [12]:
#Read all cleaned CSV files and ingest into db

def ingest_db(df,table_name,engine):
    '''This function will ingest the dataframe into database table'''
    df.to_sql(table_name, con = engine ,if_exists = 'replace' , index = False,chunksize=5000, method="multi" )


def load_raw_data():
    cleaned_folder = "data/cleaned_data"    
    for file in os.listdir(cleaned_folder):
        if file.endswith(".csv"):
            table_name = os.path.splitext(file)[0]
            file_path = os.path.join(cleaned_folder, file)
            df = pd.read_csv(file_path)
            ingest_db(df,table_name,engine)
            print(f"{table_name} loaded successfully")

if __name__ == '__main__':
    load_raw_data()

dim_customer loaded successfully
dim_date loaded successfully
dim_geography loaded successfully
dim_products loaded successfully
dim_salesperson loaded successfully
fact_sales loaded successfully


In [ ]:
#Verify the tables

In [4]:
query = """
SELECT table_name
FROM information_schema.tables
WHERE table_schema='public'
"""

pd.read_sql(query,engine)

,table_name
0,dim_customer
1,dim_date
2,dim_geography
3,dim_products
4,dim_salesperson
5,fact_sales


In [1]:
# Adding primary and forgien key

In [ ]:
# Adding forgien key for fact_sales

In [8]:
with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_fact_sales_customer
        FOREIGN KEY (customer_id)
        REFERENCES dim_customer(customer_id);
    """))

    conn.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_fact_sales_product
        FOREIGN KEY (product_id)
        REFERENCES dim_products(product_id);
    """))

    conn.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_fact_sales_salesperson
        FOREIGN KEY (salesperson_id)
        REFERENCES dim_salesperson(salesperson_id);
    """))

    conn.execute(text("""
        ALTER TABLE fact_sales
        ADD CONSTRAINT fk_fact_sales_date
        FOREIGN KEY (order_date)
        REFERENCES dim_date(date);
    """))

print("Foreign keys added successfully")

Foreign keys added successfully


In [9]:
with engine.begin() as conn:
    conn.execute(text("""
            ALTER TABLE fact_sales
            ADD CONSTRAINT fk_fact_sales_geo
            FOREIGN KEY (geo_id)
            REFERENCES dim_geography(geo_id);
        """))

In [16]:
pd.read_sql("SELECT * FROM fact_sales LIMIT 2",engine)

,order_id,order_date,customer_id,product_id,geo_id,salesperson_id,quantity,unit_price,unit_cost,discount_percent,gross_sales,discount_amount,net_sales,profit
0,O000001,2025-08-14,C04634,P0085,62,S069,32,1225.0,796.25,10,39200,3920.0,35280.0,9800.0
1,O000002,2024-06-15,C03773,P0215,41,S036,46,1892.0,1229.80,5,87032,4351.6,82680.4,26109.6


In [18]:
pd.read_sql("SELECT * FROM dim_customer LIMIT 2",engine)

,customer_id,customer_name,segment,industry
0,C00001,Olivia Brown,Retail,Retail
1,C00002,Sneha Reddy,Retail,IT


In [19]:
pd.read_sql("SELECT * FROM dim_date LIMIT 2",engine)

,date,day,day_name,month,month_name,quarter,year,month_year,year_quarter,financial_year,financial_quarter
0,2022-01-01,1,Saturday,1,January,Q1,2022,Jan-2022,2022-Q1,FY2021-22,Q4
1,2022-01-02,2,Sunday,1,January,Q1,2022,Jan-2022,2022-Q1,FY2021-22,Q4


In [20]:
pd.read_sql("SELECT * FROM dim_geography LIMIT 2",engine)

,geo_id,region,country,state,city
0,1,Asia-Pacific,India,Karnataka,Bengaluru
1,2,South America,Brazil,Sao Paulo,Sao Paulo


In [21]:
pd.read_sql("SELECT * FROM dim_products LIMIT 2",engine)

,product_id,product_name,category,brand
0,P0001,Business Switch 24 1,Networking,Dell
1,P0002,UltraView 24 1,Monitor,Cisco


In [22]:
pd.read_sql("SELECT * FROM dim_salesperson LIMIT 2",engine)

,salesperson_id,salesperson_name,designation
0,S001,James Anderson,Sales Executive
1,S002,Olivia Brown,Senior Sales Executive


### Data Validation

In [ ]:
#How many records are present in each table?

In [27]:
query = """
SELECT 'Fact_Sales' AS table_name, COUNT(*) AS total_records
FROM fact_sales

UNION ALL

SELECT 'Dim_Customer' AS table_name, COUNT(*) AS total_records
FROM dim_customer

UNION ALL

SELECT 'Dim_Product' AS table_name, COUNT(*) AS total_records
FROM dim_products

UNION ALL

SELECT 'Dim_Salesperson' AS table_name, COUNT(*) AS total_records
FROM dim_salesperson;
"""

table_counts = pd.read_sql(query, engine)

table_counts

,table_name,total_records
0,Fact_Sales,75000
1,Dim_Customer,5000
2,Dim_Product,300
3,Dim_Salesperson,100


In [ ]:
#Are there duplicate Order IDs?

In [31]:
query = """
SELECT customer_id, 
       COUNT(*) AS duplicate_count
FROM dim_customer
GROUP BY customer_id
HAVING COUNT(*) > 1
"""

duplicate_customer_id = pd.read_sql(query,engine)
duplicate_customer_id

,customer_id,duplicate_count


In [ ]:
#Are there duplicate Order IDs?

In [32]:
query = """
SELECT order_id,
       COUNT(*) AS duplicate_count
FROM fact_sales
GROUP BY order_id
HAVING COUNT(*) > 1
"""

duplicate_order_id = pd.read_sql(query,engine)
duplicate_order_id

,order_id,duplicate_count


In [ ]:
#Are there duplicate Product IDs?

In [34]:
query = """
SELECT product_id,
       COUNT(*) AS duplicate_count
FROM dim_products
GROUP BY product_id
HAVING COUNT(*) > 1
"""

duplicate_product_id = pd.read_sql(query,engine)
duplicate_product_id

,product_id,duplicate_count


In [ ]:
#Are there sales without customers

In [36]:
query = """

SELECT s.customer_id
FROM fact_sales s
LEFT JOIN dim_customer c
ON s.customer_id = c.customer_id
WHERE c.customer_id IS NULL

"""

sales_without_customers = pd.read_sql(query,engine)
sales_without_customers

,customer_id


In [ ]:
# Are there sales without products?

In [38]:
query = """
SELECT s.customer_id,
       s.product_id
FROM fact_sales s
LEFT JOIN dim_products c
ON s.product_id = c.product_id
WHERE c.product_id IS NULL

"""

sales_without_products = pd.read_sql(query,engine)

sales_without_products

,customer_id,product_id


In [49]:
query = """

SELECT
    'Customer' AS foreign_key,
    COUNT(*) AS missing_records
FROM fact_sales fs
LEFT JOIN dim_customer dc
       ON fs.customer_id = dc.customer_id
WHERE dc.customer_id IS NULL

UNION ALL

SELECT
    'Product',
    COUNT(*)
FROM fact_sales fs
LEFT JOIN dim_products dp
       ON fs.product_id = dp.product_id
WHERE dp.product_id IS NULL

UNION ALL

SELECT
    'Salesperson',
    COUNT(*)
FROM fact_sales fs
LEFT JOIN dim_salesperson ds
       ON fs.salesperson_id = ds.salesperson_id
WHERE ds.salesperson_id IS NULL

UNION ALL

SELECT
    'Geography',
    COUNT(*)
FROM fact_sales fs
LEFT JOIN dim_geography dg
       ON fs.geo_id = dg.geo_id
WHERE dg.geo_id IS NULL

UNION ALL

SELECT
    'Date',
    COUNT(*)
FROM fact_sales fs
LEFT JOIN dim_date dd
       ON fs.order_date = dd.date
WHERE dd.date IS NULL;

"""

missing_foreign_key  = pd.read_sql(query,engine)
missing_foreign_key

,foreign_key,missing_records
0,Salesperson,0
1,Geography,0
2,Product,0
3,Date,0
4,Customer,0


In [ ]:
# Are there negative quantities or prices

In [40]:
query = """

SELECT * FROM fact_sales
WHERE quantity < 0 OR unit_price < 0

"""

negative_values = pd.read_sql(query,engine)
negative_values

,order_id,order_date,customer_id,product_id,geo_id,salesperson_id,quantity,unit_price,unit_cost,discount_percent,gross_sales,discount_amount,net_sales,profit


In [ ]:
# Does Gross Sales = Quantity × Unit Price?

In [50]:
query =  """
SELECT * FROM fact_sales
WHERE gross_sales != (quantity * unit_price)
"""

incorrect_value = pd.read_sql(query,engine)
incorrect_value

,order_id,order_date,customer_id,product_id,geo_id,salesperson_id,quantity,unit_price,unit_cost,discount_percent,gross_sales,discount_amount,net_sales,profit
